# Connect Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# Find videos and create a review table

In [2]:
from pathlib import Path
import cv2
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Component03")
VIDEO_DIR = PROJECT / "input_videos"

videos = sorted(VIDEO_DIR.rglob("*.mp4"))

records = []

for video in videos:
    cap = cv2.VideoCapture(str(video))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    duration = frames / fps if fps > 0 else 0

    records.append({
        "video": str(video.relative_to(VIDEO_DIR)),
        "duration": round(duration, 2),
        "source_folder": video.parent.name
    })

df = pd.DataFrame(records)

display(df)

print("Total videos:", len(df))

,video,duration,source_folder
0,Accident/video_100.mp4,54.20,Accident
1,Accident/video_133.mp4,20.40,Accident
2,Accident/video_141.mp4,15.30,Accident
3,Accident/video_147.mp4,56.63,Accident
4,Accident/video_166.mp4,10.60,Accident
5,Accident/video_170.mp4,33.00,Accident
6,Accident/video_174.mp4,15.20,Accident
7,Accident/video_178.mp4,10.57,Accident
8,Accident/video_183.mp4,18.12,Accident
9,Accident/video_198.mp4,12.20,Accident


Total videos: 25


# Create the timestamp-annotation template

In [3]:

import csv

template_path = PROJECT / "accident_intervals.csv"

if not template_path.exists():
    with open(template_path, "w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow([
            "video",
            "start_sec",
            "end_sec",
            "label"
        ])

    print("Created:", template_path)
else:
    print("Existing review sheet preserved:", template_path)

print("Allowed labels: normal, accident, ignore")


Created: /content/drive/MyDrive/Component03/accident_intervals.csv
Allowed labels: normal, accident, ignore


# Review the 14 accident videos
Choose and play an accident video

In [40]:

from IPython.display import Video, display

accident_videos = sorted(
    (VIDEO_DIR / "Accident").glob("*.mp4")
)

for i, path in enumerate(accident_videos):
    print(i, path.name)

# Change the index to select another video
video_index = 13
selected_video = accident_videos[video_index]

print("Playing:", selected_video.name)

display(
    Video(
        str(selected_video),
        embed=True,
        width=800
    )
)


0 video_100.mp4
1 video_133.mp4
2 video_141.mp4
3 video_147.mp4
4 video_166.mp4
5 video_170.mp4
6 video_174.mp4
7 video_178.mp4
8 video_183.mp4
9 video_198.mp4
10 video_199.mp4
11 video_200.mp4
12 video_204.mp4
13 video_93.mp4
Playing: video_93.mp4


# Record the accident and normal intervals

In [41]:

import pandas as pd

# Change this index and these timestamps for each video.
video_index = 13

video_path = accident_videos[video_index]
relative_path = str(video_path.relative_to(VIDEO_DIR))

# Example ONLY – replace with timestamps you observed.
intervals = [
    (0.0, 5.0, "normal"),
    (5.0, 8.0, "accident"),
    (8.0, 10.0, "ignore")
   ]

new_rows = pd.DataFrame([
    {
        "video": relative_path,
        "start_sec": start,
        "end_sec": end,
        "label": label
    }
    for start, end, label in intervals
])

existing = pd.read_csv(template_path)

# Replace only the selected video's entries to
# avoid accidental duplicate intervals.
existing = existing[
    existing["video"] != relative_path
]

updated = pd.concat(
    [existing, new_rows],
    ignore_index=True
)

updated.to_csv(template_path, index=False)

display(updated.tail(10))
print("Saved:", template_path)


,video,start_sec,end_sec,label
45,Accident/video_200.mp4,6.0,8.0,ignore
46,Accident/video_200.mp4,8.0,10.0,accident
47,Accident/video_204.mp4,0.0,3.0,normal
48,Accident/video_204.mp4,3.0,5.0,accident
49,Accident/video_204.mp4,5.0,6.0,ignore
50,Accident/video_204.mp4,6.0,9.0,accident
51,Accident/video_204.mp4,9.0,12.0,ignore
52,Accident/video_93.mp4,0.0,5.0,normal
53,Accident/video_93.mp4,5.0,8.0,accident
54,Accident/video_93.mp4,8.0,10.0,ignore


Saved: /content/drive/MyDrive/Component03/accident_intervals.csv


# Add the 11 Normal videos

In [42]:

normal_videos = sorted(
    (VIDEO_DIR / "Normal").glob("*.mp4")
)

existing = pd.read_csv(template_path)

for video in normal_videos:
    relative_path = str(video.relative_to(VIDEO_DIR))

    duration = df.loc[
        df["video"] == relative_path, "duration"
    ].iloc[0]

    # Replace any existing intervals for this video
    existing = existing[
        existing["video"] != relative_path
    ]

    existing = pd.concat([
        existing,
        pd.DataFrame([{
            "video": relative_path,
            "start_sec": 0.0,
            "end_sec": duration,
            "label": "normal"
        }])
    ], ignore_index=True)

existing.to_csv(template_path, index=False)

print("Normal videos included:", len(normal_videos))
display(existing)


Normal videos included: 11


,video,start_sec,end_sec,label
0,Accident/video_100.mp4,0.0,6.00,normal
1,Accident/video_100.mp4,7.0,32.00,accident
2,Accident/video_100.mp4,32.0,54.00,ignore
3,Accident/video_133.mp4,0.0,1.00,normal
4,Accident/video_133.mp4,1.0,4.00,accident
...,...,...,...,...
61,Normal/20230818_acci-bg_147.mp4,0.0,10.00,normal
62,Normal/20230818_acci-bg_151.mp4,0.0,7.80,normal
63,Normal/20230818_acci-bg_153.mp4,0.0,11.96,normal
64,Normal/20230818_acci-bg_43.mp4,0.0,10.92,normal


# Validate the timestamp annotations
After recording all accident intervals, verify that every video has been reviewed, all time ranges are valid and no intervals overlap.

In [43]:

import pandas as pd

interval_df = pd.read_csv(template_path)

allowed = {"normal", "accident", "ignore"}
errors = []

for video, group in interval_df.groupby("video"):
    match = df[df["video"] == video]

    if match.empty:
        errors.append(f"Unknown video: {video}")
        continue

    duration = float(match.iloc[0]["duration"])
    group = group.sort_values("start_sec")

    previous_end = 0.0

    for _, row in group.iterrows():
        start = float(row["start_sec"])
        end = float(row["end_sec"])
        label = str(row["label"]).strip().lower()

        if not (0 <= start < end <= duration + 0.05):
            errors.append(f"Invalid interval: {video}")

        if label not in allowed:
            errors.append(f"Invalid label: {video}")

        if start < previous_end - 0.001:
            errors.append(f"Overlapping intervals: {video}")

        previous_end = max(previous_end, end)

missing_videos = set(df["video"]) - set(interval_df["video"])

print("Annotated videos:", interval_df["video"].nunique())
print("Missing videos:", len(missing_videos))
print("Errors:", len(errors))

for issue in errors[:20]:
    print(issue)

for video in sorted(missing_videos):
    print("Not reviewed:", video)

print("\nInterval counts:")
print(interval_df["label"].value_counts())


Annotated videos: 25
Missing videos: 0
Errors: 0

Interval counts:
label
normal      26
accident    23
ignore      17
Name: count, dtype: int64


# Extract the accident classification frames

Create the classification dataset

In [44]:

import json
import cv2
import shutil
from pathlib import Path

with open(PROJECT / "video_splits.json") as f:
    video_splits = json.load(f)

video_to_split = {
    video: split
    for split, paths in video_splits.items()
    for video in paths
}

ACCIDENT_DATASET = PROJECT / "accident_dataset"

# Do not automatically delete an existing dataset.
assert not ACCIDENT_DATASET.exists(), (
    "accident_dataset already exists. "
    "Back it up or rename it before rebuilding."
)

FPS_SAMPLE = 2
counts = {}

for split in ["train", "val", "test"]:
    for label in ["normal", "accident"]:
        (ACCIDENT_DATASET / split / label).mkdir(
            parents=True, exist_ok=True
        )

for video_name, group in interval_df.groupby("video"):
    video_path = VIDEO_DIR / video_name
    split = video_to_split.get(video_name)

    if split is None:
        print("Missing split:", video_name)
        continue

    cap = cv2.VideoCapture(str(video_path))
    source_fps = cap.get(cv2.CAP_PROP_FPS)

    if source_fps <= 0:
        cap.release()
        print("Invalid FPS:", video_name)
        continue

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    stride = max(1, round(source_fps / FPS_SAMPLE))

    intervals = [
        (float(r.start_sec), float(r.end_sec), r.label)
        for r in group.itertuples()
    ]

    saved = 0

    for frame_idx in range(0, total_frames, stride):
        timestamp = frame_idx / source_fps

        label = next(
            (
                lab for start, end, lab in intervals
                if start <= timestamp < end
            ),
            None
        )

        if label not in ["normal", "accident"]:
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        # Preserve source video identity in filenames
        safe_video = video_name.replace("/", "_").replace(
            "\\", "_"
        )
        filename = (
            f"{Path(safe_video).stem}"
            f"_frame{frame_idx:06d}.jpg"
        )

        output_path = (
            ACCIDENT_DATASET / split / label / filename
        )

        if cv2.imwrite(str(output_path), frame):
            saved += 1
            key = (split, label)
            counts[key] = counts.get(key, 0) + 1

    cap.release()
    print(video_name, ":", saved, "images saved")

print("\nCLASS DISTRIBUTION")
for split in ["train", "val", "test"]:
    for label in ["normal", "accident"]:
        print(split, label, counts.get((split, label), 0))


Accident/video_100.mp4 : 62 images saved
Accident/video_133.mp4 : 25 images saved
Accident/video_141.mp4 : 26 images saved
Accident/video_147.mp4 : 20 images saved
Accident/video_166.mp4 : 15 images saved
Accident/video_170.mp4 : 34 images saved
Accident/video_174.mp4 : 16 images saved
Accident/video_178.mp4 : 10 images saved
Accident/video_183.mp4 : 21 images saved
Accident/video_198.mp4 : 22 images saved
Accident/video_199.mp4 : 32 images saved
Accident/video_200.mp4 : 16 images saved
Accident/video_204.mp4 : 16 images saved
Accident/video_93.mp4 : 16 images saved
Normal/20230818_acci-bg_107.mp4 : 14 images saved
Normal/20230818_acci-bg_115.mp4 : 25 images saved
Normal/20230818_acci-bg_121.mp4 : 25 images saved
Normal/20230818_acci-bg_123.mp4 : 25 images saved
Normal/20230818_acci-bg_127.mp4 : 23 images saved
Normal/20230818_acci-bg_143.mp4 : 25 images saved
Normal/20230818_acci-bg_147.mp4 : 21 images saved
Normal/20230818_acci-bg_151.mp4 : 17 images saved
Normal/20230818_acci-bg_153

accident_dataset/
├── train/
│   ├── normal/
│   └── accident/
├── val/
│   ├── normal/
│   └── accident/
└── test/
    ├── normal/
    └── accident/